# Minimal Text Classification: TF‑IDF + XGBoost

**What this notebook does**
- Loads a CSV with columns `text` and `label`
- Cleans text minimally using `gensim`
- Creates stratified train/val/test splits
- Builds TF‑IDF features (fit only on train)
- Trains `XGBClassifier` with per-sample class weights (for imbalance)
- Evaluates with classification report and macro‑F1
- Saves model, vectorizer, and label encoder for later inference

**Input assumptions**
- Your CSV file path goes into `CSV_PATH` below.
- Two columns: `text` (string), `label` (string or int).

**Tip**: If you have GPU and XGBoost with CUDA, set `tree_method='gpu_hist'`.

In [ ]:
#!pip install -q gensim scikit-learn xgboost pandas numpy joblib
# If using conda, you can instead: conda install -c conda-forge gensim scikit-learn xgboost pandas numpy joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
import joblib
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS

CSV_PATH = 'your_dataset.csv'  # <-- change me
RANDOM_STATE = 42


In [ ]:
def clean_minimal(text: str) -> str:
    # Lowercases, de-accents, strips punctuation/numbers, drops tokens <2 by default
    toks = [t for t in simple_preprocess(str(text), deacc=True) if t not in STOPWORDS]
    return ' '.join(toks)

df = pd.read_csv(CSV_PATH)
assert {'text','label'}.issubset(df.columns), 'CSV must have columns: text,label'

# Basic hygiene
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'].str.len() > 2].dropna(subset=['text','label'])
df = df.drop_duplicates(subset=['text','label'])
print('After hygiene:', df.shape)

# Label encoding (stable mapping)
le = LabelEncoder()
y = le.fit_transform(df['label'])
X = df['text']
classes = np.unique(y)
print('Classes:', list(le.classes_))

# Stratified splits
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
)
len(X_train), len(X_val), len(X_test)


In [ ]:
# Minimal gensim cleaner for TF‑IDF
X_train_c = X_train.apply(clean_minimal)
X_val_c   = X_val.apply(clean_minimal)
X_test_c  = X_test.apply(clean_minimal)

# TF‑IDF (fit only on train to avoid leakage)
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.9,
    ngram_range=(1,2),
    max_features=100_000,
)
Xtr = tfidf.fit_transform(X_train_c)
Xva = tfidf.transform(X_val_c)
Xte = tfidf.transform(X_test_c)
Xtr.shape, Xva.shape, Xte.shape


In [ ]:
# Class weights to mitigate imbalance -> per-sample weights
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, cw)}
sample_w = np.array([class_weight_dict[int(t)] for t in y_train])
class_weight_dict


In [ ]:
# Train XGB (switch tree_method='gpu_hist' if you have CUDA build)
clf = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    tree_method='hist',
    objective='multi:softprob',
    num_class=len(classes),
    eval_metric='mlogloss',
    random_state=RANDOM_STATE,
)
clf.fit(Xtr, y_train, sample_weight=sample_w, eval_set=[(Xva, y_val)], verbose=False)
y_pred = clf.predict(Xte)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print('macro‑F1:', f1_score(y_test, y_pred, average='macro'))


In [ ]:
# Save artifacts for later inference
joblib.dump(clf, 'xgb_tfidf_model.joblib')
joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
joblib.dump(le, 'label_encoder.joblib')
print('Saved: xgb_tfidf_model.joblib, tfidf_vectorizer.joblib, label_encoder.joblib')
